# TC_01: Random forest clean baseline
Full diagnostic pipeline on clean data using RF model.
UQ: Deep Ensemble,
XAI: SHAP + LIME

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.chdir('..')
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

from src.utils.config_loader import load_config
from src.utils.data_loader import load_tabular_data
from src.data_diagnostics.quality_checks import check_tabular_quality
from src.utils.visualization import plot_class_distribution, plot_correlation, plot_feature_boxplots
from src.utils.performance_tracker import PerformanceTracker
from src.models.tabular_models import train_random_forest,evaluate_model_rf_xgb
from src.inference_diagnostics.uncertainty import deep_ensemble_prediction_sklern
from src.inference_diagnostics.explainability import shap_tab,lime_tab,calculate_consistency_tabular
from src.inference_diagnostics.flagging import assign_flags,evaluate_flags
from src.utils.visualization import plot_accuracy_comparison, plot_uncertainty_distribution, plot_flag_distribution
from src.utils.report_generator import generate_report,save_report

config = load_config()
tracker = PerformanceTracker()

### Load data and EDA

In [2]:
X_train, X_test, y_train, y_test = load_tabular_data(config)
report = check_tabular_quality(X_train, y_train, config)
plot_class_distribution(report, "Diabetes Baseline", config)
plot_correlation(report, "Diabetes Baseline", config)
plot_feature_boxplots(X_train, "Diabetes Baseline", config)

Loading dataset.
No missing value found.
Data loaded for 202944 train, 50736 test
 Features: 21
EDA started for tabular data.
Samples: 202944, Features: 21
Class distribution: {0: 174667, 1: 28277}
Imbalance ratio: 0.1619
Duplicate rows: 18580
Total outlier count: 89295
EDA completed for Diabetes Health Indicators
Saved: results/figures\class_distribution_diabetes_baseline.png
Saved: results/figures\correlation_diabetes_baseline.png
Saved: results/figures\boxplot_diabetes_baseline.png


### Train random forest baseline

In [3]:
# Random Forest
tracker.start_performance_track()
rf_model = train_random_forest(X_train, y_train, config)
tracker.stop_performance_track("Random forest training")

tracker.start_performance_track()
rf_accuracy, rf_prediction = evaluate_model_rf_xgb(rf_model, X_test, y_test)
tracker.stop_performance_track("Random forest evaluation")

Random Forest training started.
Random Forest training completed.
Random forest training: Time:1.06s, Memory:123.73MB
Model evaluation started for RF,XGB
              precision    recall  f1-score   support

           0       0.87      0.98      0.93     43667
           1       0.56      0.13      0.21      7069

    accuracy                           0.86     50736
   macro avg       0.72      0.56      0.57     50736
weighted avg       0.83      0.86      0.83     50736

Random forest evaluation: Time:0.09s, Memory:0.00MB


{'operation': 'Random forest evaluation',
 'time_seconds': 0.09,
 'memory_usage': 0}

### Uncertainty Quantification (Deep Ensembles)

In [4]:
tracker.start_performance_track()
de_means_probabilities, de_uncertainty = deep_ensemble_prediction_sklern(train_random_forest, X_train, y_train, X_test, config)
tracker.stop_performance_track("RF Deep Ensemble prediction")

de_threshold = round(float(np.percentile(de_uncertainty, 90)), 4)
de_y_prediction = de_means_probabilities.argmax(axis=1)
print(f"Deep Ensembl uncertainty status:")
print(f"Mean: {de_uncertainty.mean()}")
print(f"Max: {de_uncertainty.max()}")
print(f" 90th percentile (threshold): {de_threshold}")

plot_uncertainty_distribution(de_uncertainty, "RF Deep Ensemble", de_threshold, config)

Deep Ensemble started for tabular and sklern.
Training model with seed 0
Random Forest training started.
Random Forest training completed.
Training model with seed 1
Random Forest training started.
Random Forest training completed.
Training model with seed 2
Random Forest training started.
Random Forest training completed.
Training model with seed 3
Random Forest training started.
Random Forest training completed.
Training model with seed 4
Random Forest training started.
Random Forest training completed.
Deep Ensemble finished for tabular sklern.
RF Deep Ensemble prediction: Time:5.47s, Memory:1.35MB
Deep Ensembl uncertainty status:
Mean: 6.83106792822804e-17
Max: 1.9470490923134004e-16
 90th percentile (threshold): 0.0
Saved: results/figures\uncertainty_distribution_rf_deep_ensemble.png


### Explainability (SHAP and LIME)

In [5]:
# SHAP
tracker.start_performance_track()
shap_values, shap_samples = shap_tab(rf_model, X_train, X_test, config, is_pytorch = False)
tracker.stop_performance_track("RF SHAP")
print(f"SHAP values shape: {shap_values.shape}")

SHAP started.
SHAP finished. Explained: 50 samples.
RF SHAP: Time:15.32s, Memory:0.35MB
SHAP values shape: (50, 21, 2)


In [6]:
# LIME
tracker.start_performance_track()
lime_explanations, lime_samples = lime_tab(rf_model, X_train, X_test, config, is_pytorch = False)
tracker.stop_performance_track("RF LIME")
print(f"LIME explanations: {len(lime_explanations)}")

LIME started.


C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\prage\PycharmProjects\ml-pipeline-diagnostics-framework\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
C:\Users\pra

Explained 50 / 50 samples.
LIME finished. Explained: 50 samples.
RF LIME: Time:3.47s, Memory:0.00MB
LIME explanations: 50


In [7]:
# Calculate consistency
feature_names = list(X_train.columns)
consistency_scores = calculate_consistency_tabular(shap_values, lime_explanations, feature_names, top=5)
print(f"Mean consistency: {consistency_scores.mean()}")
print(f"Min consistency: {consistency_scores.min()}")
print(f"Max consistency: {consistency_scores.max()}")


Calculate consistency tabular.
Mean consistency: 0.72
Min consistency: 0.4
Max consistency: 1.0


### Flagging

In [9]:
# Real uncertainty value for RF is near 0 and meaning less ofr flagging system.
rf_uq_threshold = float(de_uncertainty.max()) + 1.0
rf_flags = assign_flags(de_uncertainty[:len(consistency_scores)], consistency_scores, rf_uq_threshold, 0.5)

print("RF flagging results:")
rf_flagging_result = evaluate_flags(rf_flags, rf_prediction[:len(consistency_scores)], y_test[:len(consistency_scores)])

RF flagging results:
RED: Count: 0 Accuracy:0.0%
YELLOW: Count: 2 Accuracy:100.0%
GREEN: Count: 48 Accuracy:81.2%
Flagging system is working


### Save baseline report


In [10]:
rf_baseline = {
    'model': 'Random Forest',
    'accuracy': round(rf_accuracy, 4),
    'de_uncertainty': {
        'mean': round(float(de_uncertainty.mean()), 8),
        'max': round(float(de_uncertainty.max()), 8),
        'threshold': de_threshold,
        'findings': 'Near zero uncertainty and RF internal ensemble makes external DE redundant'
    },
    'consistency':{
        'mean': round(float(consistency_scores.mean()), 4),
        'min': round(float(consistency_scores.min()), 4),
        'max': round(float(consistency_scores.max()), 4)
    },
    'flagging': rf_flagging_result,
    'performance': tracker.get_results()
}

report_output = generate_report(config,'Diabetes - RF clean baseline', stage1_result = rf_baseline )
save_report(report_output, 'TC_01_Random_Forest_Clean_Baseline.json', config)

Generating report.
Diagnostic report generated.
Saving report.
